# Dunnhumby seed44: 새 M2의 M3+M4 추가효과

**이전 seed48 두 M2 비교 노트북 대신 이 노트북만 실행합니다.** L4 GPU를 선택하고 위에서부터 실행하세요.

- 기존 M1과 M3+M4-B는 결과만 재사용합니다. 새 M2 → 새 M2+M3+M4 두 모델을 한 런타임에서 순차 학습합니다.
- N·V·q_C를 모두 유지합니다. 사용자 경제표현은 q_C·[T0+(2q_N−1)TN]·b(q_V), 상품은 b(상품 구매금액 백분위). T0=I, TN=0에서 공동학습합니다.
- ID만 그래프 전파, 경제표현은 전파 없이 최종 내적 전에 연결. 하나의 forward·optimizer, 동결·외부 재정렬 없음.
- M2: 이진 그래프와 일반 BPR. 결합: 기존 중심화 N/V M3 그래프와 split N/V M4-B 손실 그대로. N/V를 둘 다 사용합니다.
- seed44·300epoch 고정, 매25epoch 개발평가, 매epoch optimizer/RNG 포함 저장·자동재개. 최고점 선택·얼리스토핑 없음.
- ρ=.25, RBF폭=.25, ID64, 2층, lr=.0005, batch8192, ID L2=.001, uniform unseen K1, MIN_ITEM_INTER=1.
- 개발684–690일만 사용. 내부 split 이름 test는 과거 개발구간이며 보호 test698–704·holdout은 열지 않습니다. 학습에서 본 사용자–상품 쌍은 평가 정답·후보에서 제외됩니다.

## 판정 기준 — 실행 전에 고정

300epoch에서 **결합모형의 전체 사용자 가격·구매금액 가중 적중값@10 및 가중 NDCG@10이 모두 M3+M4-B와 M1을 초과**하고, 전체 Recall/NDCG@10·20·50 각각 M1의99% 이상이면 다음 검증 후보입니다.
M2 단독 미달은 결합 학습/통과를 막지 않습니다. 결합의 M3+M4 대비 정확도와 @20·@50, 모든 CLV 세그먼트·노출 지표도 함께 보고합니다. 기준 불리한 결과를 숨기거나 기준을 사후 변경하지 않습니다.

데이터 적응은 학습구간 백분위·학습되는 T0/TN과 기존 M3의 목표 가중치 CV=.20 보정에 한정됩니다. ρ 자동탐색·새 신뢰도 게이트·새 규제는 넣지 않았고, 자유 행렬이므로 ρ가 경제점수 크기의 상한은 아닙니다. H&M 검증은 이번에 하지 않습니다.
이미 사용한 개발 seed44의 탐색 결과이며 유의성·CLV 배정 귀속·최종 확증을 주장하지 않습니다.

로컬 소형 CPU 학습/재개 및 문법 검사만 완료했습니다. 실제 Drive 데이터·GPU 실행과 출력 확인은 아래에서 진행합니다.

## 1. Drive 연결·고정 코드 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = 'affeadff95569d80624c3fb654549807d3e3d01f'
REPO = Path('/content/clv-m2-basis-m5-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'GPU 런타임을 연결하세요. L4 권장.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 데이터·기존 M1/M3+M4 확인 — 학습 없음
기준 결과가 없거나 입력·설정·M4 가중치가 다르면 학습 전에 중단합니다. 기준모형을 자동 재학습하지 않습니다.

In [ ]:
import clv_m2_nv_conditional_basis_m5_screen as screen
cfg = screen.configure()
reference_dir = Path(screen.basis.ROOT + '_clv_m5_m3_m4_split_nv_s44_v1')
assert reference_dir.is_dir(), f'기존 seed44 M3+M4 결과 폴더가 필요합니다: {reference_dir}'
cfg, prepared = screen.prepare(cfg)


## 3. 새 M2 → M2+M3+M4 순차 학습
두 모델은 각자 무작위 초기화부터 공동학습합니다. M2 checkpoint를 결합에 이식하지 않습니다. 완료 모델은 캐시 재사용, 중단 모델은 마지막 완료 epoch에서 재개합니다. 평가 출력은 25epoch마다 나오며 그 사이 진행상황은 out_dir/progress에 저장됩니다.

In [ ]:
result = screen.run(cfg, prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 네 모형의 전체 사용자 성능
300epoch가 주판정이고 100/200은 설명용입니다. 모든 세그먼트·노출지표 원본은 아래 CSV/ZIP에 포함됩니다.

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
labels = {screen.M1:'M1(재사용)', screen.M5_BASE:'M3+M4-B(재사용)', screen.M2:'새 M2', screen.M5_FULL:'새 M2+M3+M4'}
absolute = result['absolute']
for epoch in (100, 200, 300):
    print(f'\n{epoch} epoch 절대지표')
    table = absolute[absolute.epoch.eq(epoch)].set_index('model_id')[metrics].T.rename(columns=labels)
    print(table.to_string())
comparison = result['comparison']
view = comparison[comparison.epoch.eq(300) & comparison.metric.isin(metrics)].copy()
view['relative_change_pct'] = 100*(view['ratio']-1)
view['model_id'] = view.model_id.map(labels)
view['reference'] = view.reference.map(labels)
print('\n300 epoch 상대 변화율(%)')
print(view.pivot(index='metric', columns=['model_id','reference'], values='relative_change_pct').to_string())
print('\nN/V 작동 진단(두 모델)')
print(result['diagnostics'].to_string(index=False))
print(json.dumps(result['paths'], ensure_ascii=False, indent=2))


## 5. 결과 ZIP 다운로드
전체 absolute·comparison·diagnostics·판정과 사전설정 JSON을 묶습니다. 큰 checkpoint는 Drive에 남습니다. M2 단독 미달도 그대로 보고하며, 결합 통과 여부는 combination_candidate를 확인하세요.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/m2_nv_conditional_basis_m5_seed44_results', 'zip', str(Path(cfg.out_dir) / 'reports'))
files.download(archive)
